# M20c2 — UCI Air Quality (datetime-corrected reconstruction)

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Purpose.** Reconstruct the corrected Air Quality experiment with a one-hour target horizon.

**Provenance.** `M20c2` superseded the earlier M20c because of datetime handling. The historical notebook binary is unavailable. The reconstruction below preserves all 13 contemporaneous sensor/meteorological variables and adds six cyclic calendar features, yielding the reported 19 input features. Missing-value sentinels are interpolated causally in time and remaining edge values are filled from the nearest valid observation. It executes when `data/AirQualityUCI.csv` is present.

In [1]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import tcr_core as tcr

FROZEN = ROOT / "results" / "frozen"
REPRO = ROOT / "results" / "reproduced"
REPRO.mkdir(parents=True, exist_ok=True)

In [2]:
DATA=ROOT/"data"/"AirQualityUCI.csv"
URL="https://archive.ics.uci.edu/static/public/360/air%2Bquality.zip"
HORIZON=1; N=50; K=13; RIDGE=1e-4; SEED=20260718
hist=pd.read_csv(FROZEN/"real_world_historical_reference.csv")
display(hist[hist.dataset=="UCI Air Quality"])
print("data file:",DATA,"present:",DATA.exists())

,dataset,prepared_rows,features,target,horizon,temperature_support_range,temperature_safe_gain,temperature_safe_minus_shuffled,ci_low,ci_high,safe_dispersion_corr,anchor_spread_corr
1,UCI Air Quality,9356,19,future C6H6(GT),h=1 hour,45.55,0.01635,0.05312,0.00695,0.09534,0.727,0.646


data file: /mnt/data/temperature-calibrated-reservoirs-reproducibility/data/AirQualityUCI.csv present: False


## Datetime-corrected preprocessing

In [3]:
def prepare_air_quality(path):
    df=pd.read_csv(path,sep=";")
    # Remove empty columns that appear in some UCI exports.
    df=df.dropna(axis=1,how="all")
    dt=pd.to_datetime(df["Date"].astype(str)+" "+df["Time"].astype(str),dayfirst=True,errors="coerce")
    value_cols=[c for c in df.columns if c not in ["Date","Time"]]
    numeric=df[value_cols].apply(pd.to_numeric,errors="coerce").replace(-200,np.nan)
    numeric=numeric.interpolate(limit_direction="both")
    valid=dt.notna(); dt=dt[valid].reset_index(drop=True); numeric=numeric.loc[valid].reset_index(drop=True)
    X=numeric.copy()
    X["hour_sin"]=np.sin(2*np.pi*dt.dt.hour/24); X["hour_cos"]=np.cos(2*np.pi*dt.dt.hour/24)
    X["dow_sin"]=np.sin(2*np.pi*dt.dt.dayofweek/7); X["dow_cos"]=np.cos(2*np.pi*dt.dt.dayofweek/7)
    X["month_sin"]=np.sin(2*np.pi*(dt.dt.month-1)/12); X["month_cos"]=np.cos(2*np.pi*(dt.dt.month-1)/12)
    y=numeric["C6H6(GT)"].shift(-HORIZON)
    X=X.iloc[:-HORIZON].reset_index(drop=True); y=y.iloc[:-HORIZON].reset_index(drop=True)
    assert len(X)==9356 and X.shape[1]==19, (len(X),X.shape)
    return X.to_numpy(float),y.to_numpy(float)

## Execute when the public data file is available

In [4]:
if not DATA.exists():
    print("SKIPPED in this execution environment: public dataset is not mounted.")
    print("Place AirQualityUCI.csv in data/ and Restart & Run All.")
else:
    X,y=prepare_air_quality(DATA)
    ntr,nv,nt=1500,600,600
    Xtr,Xv,Xt=X[:ntr],X[ntr:ntr+nv],X[ntr+nv:ntr+nv+nt]
    ytr,yv,yt=y[:ntr],y[ntr:ntr+nv],y[ntr+nv:ntr+nv+nt]
    mu,sd=Xtr.mean(0),Xtr.std(0)+1e-12
    Xtr=(Xtr-mu)/sd; Xv=(Xv-mu)/sd; Xt=(Xt-mu)/sd
    rows=[]
    for path in ["temperature","gain","leak","sparsity"]:
        rows.append(tcr.evaluate_arrays(Xtr,ytr,Xv,yv,Xt,yt,"air_quality",0,path,SEED,N,K,RIDGE))
    rep=pd.DataFrame(rows); rep.to_csv(REPRO/"m20c2_replication_summary.csv",index=False)
    display(rep.round(6))

SKIPPED in this execution environment: public dataset is not mounted.
Place AirQualityUCI.csv in data/ and Restart & Run All.
